In [1]:
import os, sys
root = os.getcwd() if os.path.isdir("src") else os.path.abspath("..")
os.chdir(root); sys.path.insert(0, root)

import pandas as pd
from sklearn.model_selection import train_test_split

elec = pd.read_csv("data/processed/reviews_electronics_clean.csv")
elec = elec.dropna(subset=["clean_text"])
print(elec.shape)

(67401, 10)


In [2]:
train, test = train_test_split(elec, test_size=0.2,
                               stratify=elec["label"], random_state=42)
print(len(train), len(test))
print(train["label"].value_counts(normalize=True).round(3))
print(test["label"].value_counts(normalize=True).round(3))

seen = set(train["clean_text"])
strict_test = test[~test["clean_text"].isin(seen)]
print("strict test size:", len(strict_test), "of", len(test))

53920 13481
label
positive    0.778
negative    0.147
neutral     0.075
Name: proportion, dtype: float64
label
positive    0.778
negative    0.147
neutral     0.075
Name: proportion, dtype: float64
strict test size: 7621 of 13481


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000)
X_train = vec.fit_transform(train["clean_text"])
X_test = vec.transform(test["clean_text"])
print(X_train.shape)

(53920, 23287)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, train["label"])

pred = clf.predict(X_test)
print("=== Normal test set ===")
print(classification_report(test["label"], pred, digits=3))

pred_s = clf.predict(vec.transform(strict_test["clean_text"]))
print("=== Strict test set (unseen texts) ===")
print(classification_report(strict_test["label"], pred_s, digits=3))

=== Normal test set ===
              precision    recall  f1-score   support

    negative      0.800     0.830     0.814      1986
     neutral      0.199     0.500     0.285      1008
    positive      0.957     0.811     0.878     10487

    accuracy                          0.791     13481
   macro avg      0.652     0.714     0.659     13481
weighted avg      0.877     0.791     0.824     13481

=== Strict test set (unseen texts) ===
              precision    recall  f1-score   support

    negative      0.761     0.815     0.787      1413
     neutral      0.227     0.450     0.301       607
    positive      0.954     0.835     0.891      5601

    accuracy                          0.801      7621
   macro avg      0.647     0.700     0.660      7621
weighted avg      0.860     0.801     0.825      7621



In [5]:
import joblib
os.makedirs("models", exist_ok=True)
joblib.dump((vec, clf), "models/baseline_tfidf_lr.pkl")

['models/baseline_tfidf_lr.pkl']